# 📊 Análisis Exploratorio de Datos (EDA) a Nivel Profesional
## Dataset: Online Retail II

Este notebook presenta un análisis en profundidad para el dataset Online Retail II, abordando limpieza de datos, análisis univariado/bivariado, patrones temporales y una base para modelos de segmentación (RFM).


### 1. Importación de Librerías y Configuración
Importamos las librerías estándar para análisis de datos y definimos paletas de colores estéticas.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configuraciones estéticas
plt.style.use('ggplot')
sns.set_theme(style="whitegrid", palette="muted")
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)


### 2. Carga del Dataset y Vista Previa
Leemos nuestro archivo CSV y visualizamos las primeras filas para comprender su estructura inicial.


In [ ]:
# Carga de datos
try:
    df = pd.read_csv('online_retail_II.csv')
    print(f"Dataset cargado correctamente. Dimensiones: {df.shape}")
except Exception as e:
    print(f"Error al cargar el dataset: {e}")

# Previsualización
display(df.head())


### 3. Inspección Estructural y Estadísticas Descriptivas
Miramos los tipos de datos, los valores faltantes y las estadísticas básicas de las columnas numéricas.


In [ ]:
# Información general sobre tipos de datos y nulos
df.info()


In [ ]:
# Estadísticas descriptivas de columnas numéricas
display(df.describe())


In [ ]:
# Estadísticas descriptivas de columnas categóricas
display(df.describe(include=['object']))


### 4. Limpieza de Datos
La limpieza de datos es crucial para un modelo de machine learning o análisis estadístico preciso. Enfrentaremos:
1. Valores nulos en `Customer ID` y `Description`.
2. Invoices cancelados (Empiezan con la letra 'C').
3. Cantidades y precios anómalos (negativos o cero).


In [ ]:
# Porcentaje de valores nulos
null_percentages = (df.isnull().sum() / len(df)) * 100
null_df = pd.DataFrame({'Valores Faltantes': df.isnull().sum(), 'Porcentaje (%)': null_percentages})
display(null_df[null_df['Valores Faltantes'] > 0].sort_values(by='Porcentaje (%)', ascending=False))

# Visualización de Nulos (Gráfico de Barras)
missing_data = null_df[null_df['Porcentaje (%)'] > 0].sort_values(by='Porcentaje (%)', ascending=False)

if not missing_data.empty:
    plt.figure(figsize=(10, 5))
    ax = sns.barplot(x=missing_data.index, y='Porcentaje (%)', data=missing_data, palette='rocket')
    plt.title("Porcentaje de Valores Nulos por Columna", fontsize=14)
    plt.ylabel("Porcentaje (%)")
    plt.xlabel("Columnas")
    
    # Añadir los porcentajes encima de cada barra
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}%', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'center', 
                    xytext = (0, 9), 
                    textcoords = 'offset points')
    plt.show()
else:
    print("No hay valores nulos en el dataset.")


In [ ]:
# Filtro 1: Eliminamos filas sin Customer ID (si requerimos análisis a nivel cliente)
# O bien, imputamos temporalmente. Para este EDA estricto, crearemos una copia limpia.
df_clean = df.dropna(subset=['Customer ID']).copy()

# Filtro 2: Excluimos cancelaciones (Invoices que inician con 'C')
canceled_mask = df_clean['Invoice'].astype(str).str.startswith('C')
print(f"Total de órdenes canceladas excluidas: {canceled_mask.sum()}")
df_clean = df_clean[~canceled_mask]

# Filtro 3: Excluimos anomalías económicas (Precio <= 0 o Cantidad <= 0)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['Price'] > 0)]

print(f"Dimensiones después de limpieza profunda: {df_clean.shape}")


### 5. Ingeniería de Características (Feature Engineering)
Creamos variables útiles para el análisis:
- **Total_Sales**: Representa el ingreso total de la fila.
- Extracción de componentes temporales (Mes, Año, Día de la Semana).


In [ ]:
# 1. Total_Sales (Revenue)
df_clean['Total_Sales'] = df_clean['Quantity'] * df_clean['Price']

# 2. Componentes temporales
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

display(df_clean[['InvoiceDate', 'Total_Sales', 'YearMonth', 'DayOfWeek']].head())


### 6. Análisis Univariado y Bivariado
Evaluamos las distribuciones clave, productos más vendidos y el impacto geográfico.


In [ ]:
# 6.1 Top 10 Países por volumen de ventas (Total Sales)
top_countries = df_clean.groupby('Country')['Total_Sales'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='Blues_r')
plt.title('Top 10 Países con Mayores Ingresos (Revenue)', fontsize=15)
plt.xlabel('Ingresos Totales (£)')
plt.ylabel('País')
plt.show()


In [ ]:
# 6.2 Top 10 Productos Más Vendidos
top_products = df_clean.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_products.values, y=top_products.index, palette='crest')
plt.title('Top 10 Productos Más Vendidos (Por Cantidad)', fontsize=15)
plt.xlabel('Cantidad Vendida')
plt.ylabel('Producto')
plt.show()


### 7. Análisis de Patrones Temporales
Descubrimos las tendencias de compra a lo largo del tiempo, los días pico y las horas con mayor tráfico.


In [ ]:
# 7.1 Tendencias de Ingresos Mensuales
monthly_sales = df_clean.groupby('YearMonth')['Total_Sales'].sum()

plt.figure(figsize=(14, 6))
monthly_sales.plot(kind='line', marker='o', color='#d62728', linewidth=2)
plt.title('Tendencia de Ingresos Mensuales', fontsize=15)
plt.xlabel('Mes')
plt.ylabel('Ingresos Totales (£)')
plt.xticks(rotation=45)
plt.grid(True)
plt.show()


In [ ]:
# 7.2 Transacciones por Día de la Semana y Hora
fig, ax = plt.subplots(1, 2, figsize=(18, 6))

# Por día
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Sunday'] # Omit Saturday if no sales
sns.countplot(data=df_clean, x='DayOfWeek', order=day_order, ax=ax[0], palette='viridis')
ax[0].set_title('Número de Transacciones por Día de la Semana')

# Por hora
sns.countplot(data=df_clean, x='Hour', ax=ax[1], palette='magma')
ax[1].set_title('Número de Transacciones por Hora del Día')

plt.show()


### 8. Análisis de Valor del Cliente (RFM)
Preparamos un análisis Recency, Frequency, Monetary, una técnica altamente profesional para comprender el valor del cliente en Retail.


In [ ]:
# Fecha actual como máximo del dataset + 1 día para cálculo de 'Recency'
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# Agrupación por cliente
rfm = df_clean.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'Invoice': 'nunique',                                    # Frequency
    'Total_Sales': 'sum'                                     # Monetary
})

rfm.rename(columns={'InvoiceDate': 'Recency', 'Invoice': 'Frequency', 'Total_Sales': 'MonetaryValue'}, inplace=True)
display(rfm.head())

# Visualizando distribuciones RFM
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(rfm['Recency'], bins=50, ax=ax[0], color='blue', kde=True).set_title('Distribución Recency')
sns.histplot(rfm[rfm['Frequency'] < 50]['Frequency'], bins=50, ax=ax[1], color='orange', kde=True).set_title('Distribución Frequency (Zoom < 50)')
sns.histplot(rfm[rfm['MonetaryValue'] < 10000]['MonetaryValue'], bins=50, ax=ax[2], color='green', kde=True).set_title('Distribución Monetary (Zoom < 10k)')
plt.tight_layout()
plt.show()


### 9. Conclusiones y Siguientes Pasos
- **Calidad de Datos:** Una importante cantidad de datos sin Customer ID indica oportunidades de mejora en el seguimiento (tracking). Las cancelaciones deben analizarse en profundidad.
- **Temporalidad:** Hay un patrón claro de mayores transacciones y volumen de ingresos a final de año, característico del Retail.
- **Siguientes Pasos (Modelado):** Los clusters RFM (como K-Means) pueden ahora aplicarse directamente al dataframe creado (`rfm`). El modelo prediciton de ventas y recomendación utilizará un set de datos libre de valores anómalos o faltantes.
